# Diagnóstico de calidad de datos: esquema RAW

Este notebook **solo revisa** los datos que Kestra cargó en Snowflake (esquema RAW). No modifica nada: cada problema encontrado se mide aquí y se corrige después en **dbt** (capas Bronze y Silver).

Se evalúan cinco dimensiones: **completitud, unicidad, validez, consistencia e integridad**.

## 1. Conexión a Snowflake

Se leen las credenciales del archivo `.env` (nunca se escriben en el código) y se abre la conexión al esquema RAW. La función `consulta()` ejecuta un SQL y devuelve el resultado como tabla de pandas.

In [3]:
import os
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv

load_dotenv("../.env")

con = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PAT"],
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database="CARTERA",
    schema="RAW",
)

def consulta(sql):
    df = con.cursor().execute(sql).fetch_pandas_all()
    df.columns = df.columns.str.lower()
    return df

consulta("select count(*) as filas from cartera_diaria")

/usr/local/python/3.14.2/lib/python3.14/site-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


,filas
0,498411


## 2. Volumen de las tablas

Cuántas filas tiene cada tabla en RAW. Sirve para dimensionar los datos y para comprobar que la ingesta cargó todo (la cartera debe tener los 31 cortes diarios de octubre).

In [4]:
for tabla in ["cartera_diaria", "operaciones", "gestiones", "pagos"]:
    n = consulta(f"select count(*) as filas from {tabla}")["filas"][0]
    print(tabla, n)

cartera_diaria 498411
operaciones 15070
gestiones 16831
pagos 22603


**Hallazgos:** las cuatro tablas se cargaron completas. La cartera tiene 498,411 filas, que corresponden a los 31 cortes diarios de octubre (unas 16 mil filas por día). Operaciones, gestiones y pagos son reportes mensuales.

## 3. Unicidad: filas duplicadas

Una fila duplicada es una fila **idéntica** a otra en todas sus columnas. Se comparan solo las columnas de datos: se excluyen `_archivo` y `_fecha_carga`, que son columnas de control agregadas por la ingesta.

In [5]:
for tabla in ["cartera_diaria", "operaciones", "gestiones", "pagos"]:
    total = consulta(f"select count(*) as n from {tabla}")["n"][0]
    unicas = consulta(f"select count(*) as n from (select distinct * exclude (_archivo, _fecha_carga) from {tabla})")["n"][0]
    print(tabla, "duplicados exactos:", total - unicas)

cartera_diaria duplicados exactos: 0
operaciones duplicados exactos: 0
gestiones duplicados exactos: 396
pagos duplicados exactos: 416


In [6]:
consulta("""
    select estado, sum(veces - 1) as duplicados
    from (
        select * exclude (_archivo, _fecha_carga), count(*) as veces
        from pagos
        group by all
        having count(*) > 1
    )
    group by estado
""")

,estado,duplicados
0,APLICADA,371
1,REVERSADA,45


**Hallazgos:**
- **Pagos:** 416 filas idénticas (1.8%). Pueden ser movimientos legítimos: un cliente puede hacer varios pagos, o tener varios reversos, del mismo valor el mismo día. El reporte no trae un identificador de transacción ni la hora, así que no se pueden distinguir. *Acción:* conservar y marcar con `es_posible_duplicado`.
- **Gestiones:** 396 filas idénticas (2.4%). Sí representan un problema de calidad: como los gestores se evalúan con indicadores de contactabilidad y seguimiento, pueden llegar a registrar gestiones duplicadas. *Acción:* marcarlas y contarlas una sola vez en los indicadores.

## 4. Grano de la cartera

El **grano** es lo que representa una fila. En la cartera, una operación puede tener varios componentes (PRINCIPAL, SEGURO, DISPOSITIVO), cada uno con su propia fila. Por eso el grano esperado es **operación + tipo de pagaré + fecha de corte**. Se comprueba que esa combinación no se repita.

In [7]:
consulta("""
    select count(*) as filas,
           count(distinct numero_operacion, tipo_pagare, fecha_corte) as combinaciones,
           count(distinct numero_operacion) as operaciones,
           count(distinct identificacion) as clientes
    from cartera_diaria
""")

,filas,combinaciones,operaciones,clientes
0,498411,498411,10185,8452


**Hallazgos:** el grano operación + tipo de pagaré + fecha se cumple (498,411 filas = 498,411 combinaciones). Hay 10,185 operaciones y 8,452 clientes.

Una operación puede tener varias filas por día por razones históricas: en un inicio, cada crédito del cliente se creaba como una operación aparte, identificada con un sufijo (DIS, SEG, LAM, etc.). Después se creó el tipo de pagaré para separar, dentro de una misma operación, el crédito principal (auto) del seguro y del dispositivo.

**Decisión de alcance:** el análisis se enfoca en los **créditos de auto**: operaciones sin sufijo y pagaré PRINCIPAL. Con eso, el grano final del análisis queda en **una fila por operación por día**.

## 5. Completitud: valores vacíos

Para cada columna se cuenta cuántas filas están vacías (nulas o con texto en blanco) y qué porcentaje representan. Solo se muestran las columnas que tienen algún vacío.

In [8]:
def vacios(tabla):
    columnas = consulta(f"""
        select column_name from information_schema.columns
        where table_schema = 'RAW' and table_name = '{tabla.upper()}'
          and left(column_name, 1) <> '_'
    """)["column_name"]
    partes = [f"count_if(trim({c}) = '' or {c} is null) as {c}" for c in columnas]
    resultado = consulta(f"select count(*) as total, {', '.join(partes)} from {tabla}")
    total = resultado.pop("total")[0]
    tabla_vacios = resultado.T
    tabla_vacios.columns = ["vacios"]
    tabla_vacios["porcentaje"] = (100 * tabla_vacios["vacios"] / total).round(2)
    return tabla_vacios[tabla_vacios["vacios"] > 0]

for tabla in ["cartera_diaria", "operaciones", "gestiones", "pagos"]:
    print(tabla)
    display(vacios(tabla))

cartera_diaria


,vacios,porcentaje
fecha_proxima_cuota,33519,6.73
fecha_primera_cuota_mora,327822,65.77
dia_pago,3116,0.63


operaciones


,vacios,porcentaje


gestiones


,vacios,porcentaje
motivo_no_pago,15912,94.54
fecha_compromiso,9608,57.09
contacto,6264,37.22


pagos


,vacios,porcentaje


**Hallazgos:** ningún vacío es un error de carga; todos tienen una explicación de negocio.
- `fecha_primera_cuota_mora` (65.8%): operaciones al día, sin cuotas en mora.
- `fecha_proxima_cuota` (6.7%): pagarés secundarios cancelados; operaciones en proceso de castigo (sufijos VEN y VIG); créditos cuyo plazo ya terminó pero siguen impagos; y créditos con siniestro, donde el pago del seguro se aplicó como abono que redujo el plazo y dejó un saldo pendiente.
- `dia_pago` (0.6%): pagarés secundarios cancelados dentro de una operación cuyo principal sigue vigente.
- `contacto` en gestiones (37.2%): no todas las respuestas exigen registrar el contacto.

*Acción:* conservar los vacíos (no imputar). Los casos de castigo (VEN y VIG) y los pagarés secundarios quedan fuera del alcance del análisis.

## 6. Validez: tipos y rangos

En RAW todo se guarda como texto. Aquí se comprueba que las fechas y los números **se puedan convertir** a su tipo correcto y que los valores tengan sentido (montos positivos, saldos no negativos, fechas dentro del período).

In [9]:
display(consulta("""
    select
        count_if(try_to_date(fecha_corte) is null) as fecha_corte_invalida,
        count_if(try_to_decimal(dias_mora, 18, 2) is null and trim(dias_mora) <> '') as dias_mora_invalido,
        count_if(try_to_decimal(capital_insoluto, 18, 2) < 0) as capital_negativo
    from cartera_diaria
"""))

display(consulta("""
    select
        count_if(try_to_date(fecha_pago) is null) as fecha_pago_invalida,
        count_if(try_to_decimal(monto, 18, 2) <= 0) as monto_no_positivo,
        count_if(try_to_date(fecha_pago) < '2026-10-01') as pagos_antes_de_octubre
    from pagos
"""))

,fecha_corte_invalida,dias_mora_invalido,capital_negativo
0,0,0,0


,fecha_pago_invalida,monto_no_positivo,pagos_antes_de_octubre
0,0,0,673


**Hallazgos:** todas las fechas y números se pueden convertir a su tipo correcto, y no hay montos ni saldos negativos. La conversión de tipos se hará en dbt (capa Bronze).

Hay **673 pagos con fecha anterior a octubre**. Es normal: algunos clientes envían su comprobante de pago después, o una conciliación bancaria reconoce pagos de meses anteriores. *Acción:* conservarlos, usando la fecha real del pago.

## 7. Consistencia y valores que definen el alcance

Se revisa que las categorías estén escritas siempre igual (espacios, abreviaciones). También se listan los **sufijos de operación** y los **tipos de pagaré**, porque son los campos que permiten aislar los créditos de auto (operación sin sufijo y pagaré PRINCIPAL).

In [11]:
display(consulta("""
    select '[' || tipo_pagare || ']' as tipo_pagare, count(*) as filas
    from cartera_diaria
    group by 1
    order by 2 desc
"""))

display(consulta("""
    select '[' || tipo_garantia || ']' as tipo_garantia, count(*) as filas
    from cartera_diaria
    group by 1
    order by 1
"""))

display(consulta("""
    select substr(numero_operacion, 10) as sufijo, count(*) as operaciones
    from operaciones
    group by 1
    order by 2 desc
"""))

,tipo_pagare,filas
0,[PRINCIPAL],307596
1,[SEGURO],113985
2,[DISPOSITIVO],76830


,tipo_garantia,filas
0,[CONTRATO DE VENTA CON RESERVA DE DOMINIO],64981
1,[FIDEICOMISO EN GARANTIA],398148
2,[PAGARE ],30422
3,[PRENDARIO],4736
4,[SIN GARANTIA],124


,sufijo,operaciones
0,,12451
1,_DISP,583
2,_VIG,522
3,_VEN,512
4,_LAM,273
5,DISP,237
6,_ADC,167
7,_MT,105
8,_SEG,98
9,SEG,37


**Hallazgos:**
- `tipo_pagare` es consistente: tres valores sin espacios (PRINCIPAL 307,596 filas; SEGURO 113,985; DISPOSITIVO 76,830).
- `tipo_garantia`: el valor PAGARE tiene espacios al final (30,422 filas). No genera categorías duplicadas, pero muestra que el sistema de origen no valida lo digitado. *Acción:* quitar espacios con `trim()` en Bronze.
- **Sufijos de operación:** existen 24 variantes, escritas sin un estándar: con y sin guion bajo (`_DISP` / `DISP`, `_SEG` / `SEG`), abreviadas (`DIS`), combinadas (`_DISP_VIG`, `_SEG_ADC`) o con guion sobrante (`_DISP_`). Esto se debe a que el código de la operación se digitaba manualmente.
- De las operaciones del histórico, 12,451 no tienen sufijo: son los créditos de auto.

*Acción:* en Silver se marca `es_auto` = operación sin sufijo. No es necesario corregir cada variante, porque toda operación con sufijo queda fuera del alcance.

## 8. Alcance del análisis

Reglas de negocio acordadas con el área de cobranzas:
- **Cartera:** solo créditos de auto (operación sin sufijo) y pagaré PRINCIPAL.
- **Gestiones:** solo créditos de auto; se excluyen las acciones BLOQUEO DISPOSITIVO y VISITA TERRENO (el foco es el call center).
- **Pagos:** solo créditos de auto, pagos APLICADOS y con motivo RECAUDACION.

Estas reglas no borran datos: en dbt se marcan en Silver y se filtran en Gold.

### 8a. Filas que quedan después de aplicar las reglas

In [19]:
consulta("""
    select 'cartera' as tabla,
           count(*) as total,
           count_if(substr(numero_operacion, 10) = '') as autos,
           count_if(substr(numero_operacion, 10) = '' and tipo_pagare = 'PRINCIPAL') as en_alcance
    from cartera_diaria

    union all

    select 'gestiones',
           count(*),
           count_if(substr(numero_operacion, 10) = ''),
           count_if(substr(numero_operacion, 10) = ''
                    and accion not in ('BLOQUEO DISPOSITIVO', 'VISITA TERRENO'))
    from gestiones

    union all

    select 'pagos',
           count(*),
           count_if(substr(numero_operacion, 10) = ''),
           count_if(substr(numero_operacion, 10) = '' and estado = 'APLICADA' and motivo = 'RECAUDACION')
    from pagos
""")

,tabla,total,autos,en_alcance
0,cartera,498411,452070,261255
1,gestiones,16831,15838,13777
2,pagos,22603,21870,12733


**Hallazgos:**

| Tabla | Total | Solo autos | En alcance | % que queda |
|---|---|---|---|---|
| Cartera | 498,411 | 452,070 | 261,255 | 52% |
| Gestiones | 16,831 | 15,838 | 13,777 | 82% |
| Pagos | 22,603 | 21,870 | 21,244 | 94% |

- **Cartera:** la regla que más reduce es quedarse solo con el pagaré PRINCIPAL: dentro de las operaciones de auto, el seguro y el dispositivo son ~191 mil filas.
- **Gestiones:** BLOQUEO DISPOSITIVO y VISITA TERRENO representan 2,061 gestiones (13% de las de autos), que no corresponden al call center.
- **Pagos:** 626 pagos de autos están reversados y no se cuentan como recaudo.

### 8b. Motivos de los pagos aplicados
Solo el motivo RECAUDACION corresponde a dinero recaudado por la gestión de cobranza.

In [20]:
consulta("""
    select motivo, count(*) as pagos, sum(try_to_decimal(monto, 18, 2)) as monto
    from pagos
    where substr(numero_operacion, 10) = '' and estado = 'APLICADA'
    group by 1
    order by 2 desc
""")

,motivo,pagos,monto
0,RECAUDACION,12733,5081685.46
1,PAGO CUOTAS VENTA DE CARTERA,7612,4856866.99
2,CONDONACION AUTOMATICA,495,1300.36
3,CANCELACIÓN CLONADA,145,1607473.46
4,CRUCE CUENTA CONTABLE,99,103689.81
5,CONDONACION MANUAL,90,47375.79
6,ANULACIÓN DISPOSITIVO,29,25386.30
7,ANULACIÓN SEGURO,26,13539.99
8,CANCELACIÓN NEGOCIO CAÍDO,15,190742.55


**Hallazgos:** de los 21,244 pagos aplicados de autos, solo 12,733 tienen motivo RECAUDACION (42.6% del monto). Los demás motivos (venta de cartera, cancelación clonada, condonaciones, cruces contables y anulaciones) no corresponden a dinero recaudado por la gestión de cobranza. *Acción:* marcar `es_recaudacion` en Silver y filtrar en Gold.

### 8c. Estados del histórico de operaciones
El histórico se usa solo para identificar operaciones **CANCELADAS**, que explican por qué una operación deja de aparecer en la cartera diaria. Los demás estados (ACTIVO, VENCIDO, NON ACCRUAL, CASTIGADA) ya vienen día a día en la cartera.

In [21]:
consulta("""
    select estado_actual, count(*) as operaciones
    from operaciones
    group by 1
    order by 2 desc
""")

,estado_actual,operaciones
0,ACTIVO,6473
1,CANCELADO,3513
2,NON ACCRUAL,2270
3,GENERADO,1364
4,VENCIDO,1174
5,CASTIGADA,276


**Hallazgos:** 3,513 operaciones del histórico están CANCELADAS. En Gold se marcan con `es_cancelada` en la dimensión de operaciones. El estado GENERADO (1,364 operaciones) no se utiliza, porque su significado (crédito nuevo aún no activado o negocio no concretado) requiere validación con el área responsable.

## 9. Precisión e integridad: ¿las tablas cruzan entre sí?

Se verifica si cada pago y cada gestión corresponde a una operación que existe en el histórico de operaciones.

In [14]:
consulta("""
    select 'pagos' as tabla,
           count(*) as filas,
           count_if(numero_operacion not in (select numero_operacion from operaciones)) as sin_operacion
    from pagos

    union all

    select 'gestiones',
           count(*),
           count_if(numero_operacion not in (select numero_operacion from operaciones))
    from gestiones
""")

,tabla,filas,sin_operacion
0,pagos,22603,7908
1,gestiones,16831,0


**Hallazgos:**
- **Gestiones:** todas corresponden a operaciones del histórico (0 sin operación). La integridad es completa.
- **Pagos:** 7,908 de 22,603 pagos (35%) son de operaciones que no aparecen en el histórico de operaciones. El histórico es una foto tomada en noviembre y no tiene fecha: puede no incluir operaciones ya canceladas (por ejemplo, con una precancelación) o generadas en meses anteriores.

*Acción:* conservar estos pagos, porque son dinero realmente recaudado. En Gold, la dimensión de operaciones se construye con las operaciones de **todas** las tablas, no solo del histórico, para que ningún pago quede sin su operación.

## 10. Resumen: problemas de calidad y acciones

| # | Dimensión | Problema | Evidencia | Acción (dónde) | Justificación |
|---|---|---|---|---|---|
| 1 | Completitud | `fecha_primera_cuota_mora` vacía | 65.8% de la cartera | Conservar, no imputar (Bronze) | Las operaciones al día no tienen cuota en mora |
| 2 | Completitud | `fecha_proxima_cuota` vacía | 6.7% de la cartera | Conservar (Bronze) | Pagarés secundarios cancelados, castigos (VEN/VIG), plazos vencidos impagos y siniestros con plazo reducido |
| 3 | Completitud | `dia_pago` vacío | 0.6% de la cartera | Conservar (Bronze) | Pagarés secundarios cancelados con principal vigente |
| 4 | Completitud | `contacto` vacío | 37.2% de las gestiones | Reemplazar por "NO REGISTRADO" (Silver) | No todas las respuestas exigen contacto; se distingue de un dato perdido |
| 5 | Unicidad | Pagos idénticos | 416 filas (371 aplicados, 45 reversados) | Conservar y marcar `es_posible_duplicado` (Silver) | Pueden ser pagos o reversos legítimos del mismo valor el mismo día; el reporte no trae id de transacción ni hora |
| 6 | Unicidad | Gestiones idénticas | 396 filas (2.4%) | Marcar `es_posible_duplicado` (Silver) y contar una sola vez (Gold) | Los gestores pueden duplicar registros para mejorar sus indicadores de contactabilidad |
| 7 | Validez | Todos los campos llegan como texto | 0 valores inválidos en fechas y montos | Convertir tipos con `try_to_*` (Bronze) | Permite operar fechas y montos con su tipo correcto |
| 8 | Validez | Pagos con fecha anterior a octubre | 673 pagos | Conservar (Silver) | Comprobantes enviados después o conciliación bancaria |
| 9 | Consistencia | `tipo_garantia` con espacios al final | 30,422 filas (PAGARE) | Quitar espacios con `trim()` (Bronze) | El sistema de origen no valida lo digitado |
| 10 | Consistencia | Sufijos de operación sin estándar | 24 variantes (`_DISP` / `DISP` / `DIS`, combinados) | Marcar `es_auto` = sin sufijo (Silver) | El código se digitaba manualmente; cualquier sufijo queda fuera del alcance |
| 11 | Precisión | Pagos de operaciones que no están en el histórico | 7,908 pagos (35%); gestiones: 0 | Conservar; la dimensión de operaciones se arma con todas las tablas (Gold) | El histórico es una foto de noviembre sin fecha: no incluye operaciones canceladas o antiguas |
| 12 | Alcance | Cartera fuera del foco (con sufijo o pagaré no PRINCIPAL) | Quedan 261,255 de 498,411 filas (52%) | Marcar `es_auto` y `es_principal` (Silver), filtrar (Gold) | El análisis se enfoca en créditos de auto |
| 13 | Alcance | Acciones BLOQUEO DISPOSITIVO y VISITA TERRENO | 2,061 gestiones de autos | Marcar `es_call_center` (Silver), filtrar (Gold) | El foco son los gestores de call center |
| 14 | Alcance | Pagos reversados | 626 pagos de autos | Marcar `es_aplicado` (Silver), filtrar (Gold) | Un reverso no es dinero recaudado |
| 15 | Alcance | Pagos aplicados que no son de recaudación | 8,511 pagos de autos (57% del monto) | Marcar `es_recaudacion` (Silver), filtrar (Gold) | Venta de cartera, condonaciones, cruces y anulaciones no son recaudo de la gestión de cobranza |
| 16 | Alcance | Estado GENERADO del histórico con significado ambiguo | 1,364 operaciones | No se utiliza; solo se marca `es_cancelada` (Gold) | Puede ser un crédito nuevo o un negocio no concretado; requiere validación con el área responsable |

In [15]:
consulta("""
    select count(*) as filas,
           count(fecha_corte) as fecha_corte,
           count(dias_mora) as dias_mora,
           count(capital_insoluto) as capital_insoluto,
           count(dia_pago) as dia_pago
    from CARTERA.BRONZE.STG_CARTERA_DIARIA
""")

,filas,fecha_corte,dias_mora,capital_insoluto,dia_pago
0,498411,498411,498411,498411,495295


**Verificación de Bronze:** después de convertir los tipos, las columnas conservan todos sus valores (498,411 filas en fecha, días de mora y capital). `dia_pago` tiene 495,295 valores, porque los 3,116 vacíos ya venían así desde RAW. Ninguna conversión perdió datos.